In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取数据
print("Loading data...")
data = pd.read_csv('melb_data.csv')

y = data.Price

# 为了简化问题，我们只采用数值列作为特征进行训练
x = data.drop(['Price'], axis=1)

# 将原始数据划分为训练集和验证集
x_train_full, x_valid_full, y_train, y_valid = train_test_split(x, y, train_size=0.8, test_size=0.2, random_state=0)

print("Setup complete")

Loading data...
Setup complete


In [ ]:
# 数据预处理
# 采用最简单的方法，移除含有缺失值的列
cols_with_missing = [col for col in x_train_full.columns if x_train_full[col].isnull().any()] 
x_train_full.drop(cols_with_missing, axis=1, inplace=True)
x_valid_full.drop(cols_with_missing, axis=1, inplace=True)

# 计算对应列唯一值的出现次数，正对应着教程所说的：
# 如果分类变量取值很多，独热编码通常表现不佳（即，通常不会用于取值超过 15 个不同值的变量）。
# PS：使用 Kaggle 上的代码我没有跑出 ['Type', 'Method', 'Regionname']，我是将下面这行代码中的'object'改为'string'才能正确得到结果。
low_cardinality_cols = [cname for cname in x_train_full.columns if x_train_full[cname].nunique() < 10 and x_train_full[cname].dtype == 'string']
# 选取数值列
numerical_cols = [cname for cname in x_train_full.columns if x_train_full[cname].dtype in ['int64', 'float64']]
# 拼接得到对应列
my_cols = low_cardinality_cols + numerical_cols
x_train = x_train_full[my_cols].copy()
x_valid = x_valid_full[my_cols].copy()

print(x_train.head())

      Type Method             Regionname  Rooms  Distance  Postcode  Bedroom2  \
12167    u      S  Southern Metropolitan      1       5.0    3182.0       1.0   
6524     h     SA   Western Metropolitan      2       8.0    3016.0       2.0   
8413     h      S   Western Metropolitan      3      12.6    3020.0       3.0   
2919     u     SP  Northern Metropolitan      3      13.0    3046.0       3.0   
6043     h      S   Western Metropolitan      3      13.3    3020.0       3.0   

       Bathroom  Landsize  Lattitude  Longtitude  Propertycount  
12167       1.0       0.0  -37.85984    144.9867        13240.0  
6524        2.0     193.0  -37.85800    144.9005         6380.0  
8413        1.0     555.0  -37.79880    144.8220         3755.0  
2919        1.0     265.0  -37.70830    144.9158         8870.0  
6043        1.0     673.0  -37.76230    144.8272         4217.0  


In [13]:
# 获取分类变量的对应列
s = (x_train.dtypes == 'string')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Categorical variables:
['Type', 'Method', 'Regionname']


In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 我们利用 MAE 检验不同处理方法的好坏
def score_dataset(x_train, x_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(x_train, y_train)
    preds = model.predict(x_valid)
    return mean_absolute_error(y_valid, preds)

# 1.我们使用 select_dtypes() 方法删除 object 列。
drop_X_train = x_train.select_dtypes(exclude=['object'])
drop_X_valid = x_valid.select_dtypes(exclude=['object'])

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

# 2.使用序数编码处理
from sklearn.preprocessing import OrdinalEncoder

# 复制一份数据，保持原始数据不变
label_X_train = x_train.copy()
label_X_valid = x_valid.copy()

ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(x_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(x_valid[object_cols])

print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

# 3.使用 OneHot 编码处理
from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(x_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(x_valid[object_cols]))

# 将索引值赋值回
OH_cols_train.index = x_train.index
OH_cols_valid.index = x_valid.index

# 移除原本的类别变量列，我们使用OneHot编码列代替
num_X_train = x_train.drop(object_cols, axis=1)
num_X_valid = x_valid.drop(object_cols, axis=1)

# 将编码列添加回数值列
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
183550.22137772635
MAE from Approach 2 (Ordinal Encoding):
175062.2967599411
MAE from Approach 3 (One-Hot Encoding):
176703.63810751104
